# Week 15: other transforms and the time-series project

**What we do.** We meet three further transforms from the syllabus — the Radon transform (computed tomography), the Hough transform (finding straight lines) and the Zak transform (time-frequency analysis on a lattice) — and then start the course project: a multiscale analysis of a price series with the wavelet transform, following chapter 7 of the lecture notes.

**What to expect.** A tomographic image can be reconstructed from its projections; the lines of a picture become bright points in the Hough accumulator; the Zak transform is invertible. A price series splits into a long-term trend, medium-term cycles and day-to-day noise, and two series can be correlated on one scale and independent on another.

**How to read the code.** Functions come from `wavelets_lab.transforms` (Radon, Hough, Zak), `wavelets_lab.edges` (Canny edges for Hough), `wavelets_lab.images` (loading and display) and `wavelets_lab.timeseries` (the project).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import edges, images, timeseries, transforms

## The Radon transform and filtered back-projection

**What we do.** We create a simple "phantom" (an ellipse with a darker rectangle inside), compute its Radon transform (sinogram) for the angles 0°–179° and reconstruct it with filtered back-projection.

**What to expect.** Each column of the sinogram is one projection (the integrals of the image along parallel lines); edges of the phantom trace sine-like curves. The reconstruction reproduces the phantom with small streak artefacts.

**How to read the code.** `transforms.radon(phantom)` returns an array with one column per angle; we label its axes "projection angle" and "detector position". `transforms.inverse_radon(sinogram)` performs the reconstruction. The phantom is built with a Boolean expression of the coordinates from `np.mgrid`.

In [ ]:
size = 128
rows, cols = np.mgrid[:size, :size] - size / 2
phantom = ((rows / 40) ** 2 + (cols / 25) ** 2 <= 1).astype(float)
phantom[50:60, 55:75] = 0.5
sinogram = transforms.radon(phantom)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
images.show_image(phantom, "phantom", ax=axes[0], scaled=True)
images.show_image(
    sinogram,
    "sinogram (Radon transform)",
    ax=axes[1],
    scaled=True,
    xlabel="projection angle [deg]",
    ylabel="detector position [pixel]",
)
images.show_image(
    transforms.inverse_radon(sinogram),
    "filtered back-projection",
    ax=axes[2],
    scaled=True,
)
plt.tight_layout()
plt.show()

## The Hough transform of Canny edges

**What we do.** We detect the edges of the checkerboard image with Canny (week 3) and compute the Hough transform of the edge map.

**What to expect.** The checkerboard consists of horizontal and vertical lines, so the accumulator has bright points at the angles −90° (or 90°) and 0°, one for every line, at the distances `ρ` of the lines. The four strongest lines printed below have `θ` close to 0 or ±π/2.

**How to read the code.** `transforms.hough_lines(edge_map)` returns the vote counts, the angles (radians) and the distances `ρ`; `imshow` with `extent` puts the angles in degrees on the horizontal axis and `ρ` on the vertical one. `transforms.strongest_lines(..., 4)` returns `(θ, ρ)` of the four lines with most votes.

In [ ]:
checker = images.load_test_image("checkerboard")
edge_map = edges.canny(checker, 20, 50)
accumulator, angles, rhos = transforms.hough_lines(edge_map)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
images.show_image(edge_map, "Canny edges", ax=axes[0], scaled=True)
axes[1].imshow(accumulator, aspect="auto", extent=(-90, 90, rhos[-1], rhos[0]))
axes[1].set_xlabel("angle θ [deg]")
axes[1].set_ylabel("distance ρ [pixel]")
axes[1].set_title("Hough accumulator")
plt.tight_layout()
plt.show()
transforms.strongest_lines(accumulator, angles, rhos, 4)

## The Zak transform

**What we do.** We compute the discrete Zak transform of a cosine of length 64 with period `L = 8`, look at its magnitude, and invert it.

**What to expect.** The transform is an 8×8 array (`L × M` with `M = 64 / 8`). The cosine makes exactly 3 oscillations in 64 samples, so along the second axis the energy is concentrated at the "frequencies" 3 and `M - 3 = 5`. The inverse transform recovers the signal up to rounding.

**How to read the code.** `transforms.zak(x, period=8)` reshapes the signal into 8 rows of every 8th sample and takes the FFT of each row; `transforms.inverse_zak` undoes this. The magnitude is shown with `show_image` with suitable axis labels.

In [ ]:
x = np.cos(2 * np.pi * 3 * np.arange(64) / 64)
z = transforms.zak(x, period=8)
fig, ax = plt.subplots(figsize=(5, 4.5))
images.show_image(
    np.abs(z),
    "|Zak transform|",
    ax=ax,
    scaled=True,
    xlabel="frequency index k",
    ylabel="time index n",
)
plt.show()
z.shape, np.abs(transforms.inverse_zak(z) - x).max()

## Project: a price series across scales

**What we do.** We generate a synthetic daily price series (2048 trading days) and split its logarithm into seven wavelet components (db4, 6 levels), which we group into large, medium and small time scales.

**What to expect.** The "large" component (trend and slowest detail, scales of about a year and more) follows the slow cycle of the price; "medium" (scales of weeks to months) shows the monthly oscillations; "small" (scales of days) looks like noise. The three components add up exactly to the series.

**How to read the code.** `timeseries.synthetic_prices` returns the prices; `timeseries.multiscale_components(log_price, "db4", 6)` returns `[smooth, d_6, d_5, …, d_1]`, where `d_j` describes fluctuations over about `2^j` days. `timeseries.group_scales` adds the components listed in the dictionary `groups`.

In [ ]:
prices = timeseries.synthetic_prices(2048, rng=0)
log_price = np.log(prices)
components = timeseries.multiscale_components(log_price, "db4", 6)
groups = {
    "large (> 64 days)": [0, 1],
    "medium (8-64 days)": [2, 3, 4],
    "small (< 8 days)": [5, 6],
}
scales = timeseries.group_scales(components, groups)
days = np.arange(prices.size)
fig, axes = plt.subplots(4, 1, figsize=(10, 11), sharex=True)
axes[0].plot(days, log_price)
axes[0].set_title("log price")
for ax, (name, component) in zip(axes[1:], scales.items(), strict=True):
    ax.plot(days, component)
    ax.set_title(name)
for ax in axes:
    ax.set_xlabel("day")
    ax.set_ylabel("log price component")
plt.tight_layout()
plt.show()
np.abs(sum(scales.values()) - log_price).max()

## Correlations scale by scale

**What we do.** We build a second series whose daily returns are half the returns of the first series plus independent returns of its own, and correlate the two return series on every scale.

**What to expect.** The correlation coefficients (between −1 and 1) are clearly positive on all scales, because of the common component; with real data the correlation of two assets usually differs between scales. The bar chart shows one correlation per wavelet component.

**How to read the code.** `np.diff(np.log(prices), prepend=...)` computes daily log returns (same length as the prices). `timeseries.scale_correlations(a, b, "db4", 6)` decomposes both series and returns the Pearson correlation of each pair of components.

In [ ]:
other = timeseries.synthetic_prices(2048, rng=1)
returns_a = np.diff(log_price, prepend=log_price[0])
returns_b = 0.5 * returns_a + np.diff(np.log(other), prepend=np.log(other[0]))
correlations = timeseries.scale_correlations(returns_a, returns_b, "db4", 6)
names = ["smooth"] + [f"d{j}" for j in range(6, 0, -1)]
plt.figure(figsize=(8, 3.5))
plt.bar(names, correlations)
plt.xlabel("wavelet component (coarse to fine)")
plt.ylabel("correlation coefficient")
plt.ylim(-1, 1)
plt.show()
np.round(correlations, 3)

## Project task

**What we do.** You replace the synthetic data with real data and interpret the results.

**What to expect.** Choose two daily price series (for example two commodities, or a commodity and an exchange rate, as in chapter 7 of the lecture notes), cut them to a common period whose length is divisible by `2^level`, and repeat the two analyses above. Describe on which scales the series are related and what that means economically.

**How to read the code.** Read the data with `numpy.loadtxt` or `pandas.read_csv`, keep the closing prices as a NumPy array, and call the same functions of `wavelets_lab.timeseries`. No new functions are needed.

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** Four modules are used: `wavelets_lab.transforms` (Radon, Hough, Zak), `wavelets_lab.edges` (Canny, with its internal steps), `wavelets_lab.images` (loading and display) and `wavelets_lab.timeseries` (the project). Canny itself also uses `filters.gaussian_filter`, described in the notebook of week 3.

**How to read the code.** Functions marked *used internally* are called by the others.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `np.mgrid[:n, :n]` | grids of row and column indices | slices → two 2D arrays |
| `ax.imshow(a, aspect, extent)` | shows an array as an image with given axis ranges | 2D array → image |
| `np.diff(a, prepend=a0)` | differences of consecutive values | array → array of the same length |
| `np.log`, `np.round` | natural logarithm, rounding | array → array |
| `plt.bar(labels, heights)` | bar chart | lists → bars |

### Module `wavelets_lab.transforms` (file `src/wavelets_lab/transforms.py`)

Other transforms from the syllabus: the Radon transform (tomography), the Hough transform (line detection) and the discrete Zak transform (time-frequency analysis on a lattice). Imports:
```python
import numpy as np
from skimage import transform
```

#### `transforms.radon(image, angles=None)`

**What it is.** The Radon transform: for every angle `θ` the image is integrated along parallel lines perpendicular to the direction `θ`, giving one projection (a column). All projections together form the **sinogram** (a point of the image draws a sine curve in it). This is what a CT scanner measures. Computed with `skimage.transform.radon`.

**Input.** `image` (array-like): greyscale image, ideally zero outside the inscribed circle; `angles` (array-like or `None`): projection angles in degrees, `0, 1, …, 179` by default.

**Output.** `float64` sinogram: rows = position along the detector (pixels), columns = angles.

**How to write it yourself.**
1. `theta = np.arange(180.0)` if `angles is None`, else the given angles.
2. Return `skimage.transform.radon(np.asarray(image, dtype=np.float64), theta=theta)`.

#### `transforms.inverse_radon(sinogram, angles=None)`

**What it is.** Reconstruction of the image from its sinogram by filtered back-projection (`skimage.transform.iradon`): each projection is filtered with a ramp filter in the Fourier domain and "smeared back" over the image along its lines.

**Input.** `sinogram` (array-like), `angles` (array-like or `None`): the angles used in `radon` (by default one per column: `0, 1, …`).

**Output.** `float64` reconstructed image.

**How to write it yourself.**
1. `theta = np.arange(float(sinogram.shape[1]))` if `angles is None`, else the given angles.
2. Return `skimage.transform.iradon(sinogram, theta=theta)`.

#### `transforms.hough_lines(edges, n_angles=180)`

**What it is.** The Hough transform for straight lines. A line is described by its normal form `ρ = x cos θ + y sin θ` (`x` = column, `y` = row, `ρ` = signed distance from the origin, `θ` = direction of the normal). Every edge pixel votes for all lines through it, i.e. for one `ρ` per angle; lines present in the image collect many votes and appear as bright points of the accumulator.

**Input.** `edges` (array-like): Boolean edge map (e.g. from `edges.canny`); `n_angles` (`int`): number of angles in `[-90°, 90°)`.

**Output.** Tuple `(accumulator, angles, rhos)`: integer vote counts of shape `(len(rhos), n_angles)`, angles in radians, and distances `ρ` in pixels from `-D` to `D` (`D` = image diagonal).

**How to write it yourself.**
1. `angles = np.deg2rad(np.linspace(-90, 90, n_angles, endpoint=False))`.
2. `D = int(np.ceil(np.hypot(rows, cols)))`; `rhos = np.arange(-D, D + 1)`; `accumulator = np.zeros((len(rhos), len(angles)), dtype=int)`.
3. Coordinates of edge pixels: `r, c = np.nonzero(edges)`.
4. Votes of all pixels for all angles at once: `votes = np.rint(np.outer(c, np.cos(angles)) + np.outer(r, np.sin(angles))).astype(int)`; shape `(pixels, angles)`.
5. For every angle index `j`: `np.add.at(accumulator[:, j], votes[:, j] + D, 1)` (`+ D` turns `ρ` into a row index; `np.add.at` counts repeated indices correctly).

#### `transforms.strongest_lines(accumulator, angles, rhos, count=1)`

**What it is.** Reads off the parameters of the lines with the most votes.

**Input.** The three outputs of `hough_lines` and `count` (`int`): how many lines.

**Output.** List of `(theta, rho)` pairs (radians, pixels), strongest first.

**How to write it yourself.**
1. Indices of the largest entries of the flattened array: `order = np.argsort(accumulator, axis=None)[::-1][:count]`.
2. Back to 2D indices: `rho_idx, angle_idx = np.unravel_index(order, accumulator.shape)`.
3. Return `[(angles[a], rhos[r]) for r, a in zip(rho_idx, angle_idx)]` as floats.

#### `transforms.zak(x, period)`

**What it is.** The discrete Zak transform. For a signal of length `N = L M` with `L = period`, `Z[n, k] = sum_{l=0}^{M-1} x[n + lL] e^{-2πi kl/M}`: the samples `x[n], x[n + L], x[n + 2L], …` (one per period) are Fourier transformed. It is a time-frequency representation on an `L × M` lattice and is the basic tool for analysing Gabor (windowed Fourier) systems.

**Input.** `x` (array-like): signal of length divisible by `period`; `period` (`int`): `L`.

**Output.** Complex array of shape `(L, M)`. Raises `ValueError` if the length is not divisible.

**How to write it yourself.**
1. Check `len(x) % period == 0`.
2. `blocks = x.reshape(-1, period).T`: row `n` is `x[n], x[n + L], …` (shape `(L, M)`).
3. Return `np.fft.fft(blocks, axis=1)`.

#### `transforms.inverse_zak(coefficients)`

**What it is.** The inverse of `zak`: inverse FFT along the rows, then undo the reshaping.

**Input.** `coefficients` (array-like): Zak transform of shape `(L, M)`.

**Output.** Complex array of length `L M` (take `.real` for a real signal).

**How to write it yourself.**
1. `blocks = np.fft.ifft(coefficients, axis=1)`.
2. Return `blocks.T.reshape(-1)` (transpose back to rows of length `L`, then flatten).

### Module `wavelets_lab.edges` (file `src/wavelets_lab/edges.py`)

Image gradients and the Canny edge detector, following the steps of the lecture notes. Imports:
```python
import numpy as np
from scipy import ndimage
from wavelets_lab.filters import convolve2d, gaussian_filter
```
Constants: the Sobel masks `SOBEL_X = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]` (derivative along columns, i.e. in the x direction) and `SOBEL_Y = SOBEL_X.T` (derivative along rows, y direction, rows grow downwards). The notes call such masks Prewitt-type masks; Sobel masks additionally weight the middle row by 2.

#### `edges.canny(image, low=20.0, high=50.0, smoothing=5)`

**What it is.** The complete Canny edge detector.

**Input.**
- `image` (array-like): greyscale image on `[0, 255]`.
- `low`, `high` (`float`): hysteresis thresholds for the gradient magnitude.
- `smoothing` (`3` or `5`): size of the Gaussian mask used first.

**Output.** Boolean edge map of the image's shape.

**How to write it yourself.**
1. Smooth: `smoothed = gaussian_filter(image, size=smoothing)` (reduces noise, which would otherwise produce false edges).
2. Gradient: `gx, gy = gradient(smoothed)`.
3. Thin: `thin = non_maximum_suppression(gradient_magnitude(gx, gy), quantize_direction(gx, gy))`.
4. Select: return `hysteresis_threshold(thin, low, high)`.

#### `edges.gradient(image)`

**What it is.** Estimates the two partial derivatives of the image intensity with the Sobel masks.

**Input.**
- `image` (array-like): greyscale image.

**Output.** Tuple `(gx, gy)` of `float64` arrays with the image's shape: `gx` is the derivative from left to right, `gy` from top to bottom. For the ramp `image[i, j] = j` the Sobel estimate is `gx = 8` (the mask sums 1+2+1 differences of 2).

**How to write it yourself.**
1. Define `SOBEL_X` and `SOBEL_Y` as above.
2. We want correlation (mask applied as written), but `convolve2d` flips the mask; so pass a flipped mask to undo the flip: `gx = convolve2d(image, SOBEL_X[::-1, ::-1])`.
3. The same for `gy` with `SOBEL_Y`.
4. Return `gx, gy`.

#### `edges.gradient_magnitude(gx, gy)`

**What it is.** The length of the gradient vector, `sqrt(gx**2 + gy**2)`, i.e. how fast the intensity changes. It is large on edges.

**Input.** `gx`, `gy` (array-like): outputs of `gradient`.

**Output.** `float64` array.

**How to write it yourself.** Return `np.hypot(gx, gy)` (a numerically safe `sqrt(gx**2 + gy**2)`), converted to `float64`.

#### `edges.quantize_direction(gx, gy)`

**What it is.** Rounds the direction of the gradient to one of four directions: 0 = 0° (horizontal gradient, i.e. a vertical edge), 1 = 45°, 2 = 90°, 3 = 135°. Directions differing by 180° are treated as the same.

**Input.** `gx`, `gy` (array-like): outputs of `gradient`.

**Output.** Integer array with values 0, 1, 2, 3.

**How to write it yourself.**
1. Compute the angle in degrees with `np.degrees(np.arctan2(-gy, gx))`. The minus sign is needed because rows grow downwards, so "up" in the picture is negative `y`; with `-gy` angles are counted counter-clockwise as usual.
2. Reduce modulo 180 (`% 180.0`) to identify opposite directions.
3. Divide by 45, round with `np.rint`, convert to `int` and take `% 4` (so that 180° becomes 0 again).

#### `edges.non_maximum_suppression(magnitude, direction)`

**What it is.** Thins edges to a width of one pixel: a pixel is kept only if its gradient magnitude is at least as large as that of both neighbours along the gradient direction; all other pixels are set to 0.

**Input.**
- `magnitude` (array-like): gradient magnitude.
- `direction` (array-like): quantized directions from `quantize_direction`.

**Output.** `float64` array: the magnitude where it is a local maximum, 0 elsewhere.

**How to write it yourself.**
1. List the neighbour offsets `(d_row, d_col)` for the four directions: `((0, 1), (-1, 1), (-1, 0), (-1, -1))` for 0°, 45°, 90°, 135° (45° points up-right, so the row decreases).
2. Pad the magnitude with one row/column of zeros: `padded = np.pad(mag, 1, mode="constant")`.
3. For each direction index `k` with offset `(dr, dc)`: the neighbour in front of every pixel is the shifted slice `padded[1 + dr : 1 + dr + rows, 1 + dc : 1 + dc + cols]`, the neighbour behind is the slice with `-dr, -dc`.
4. Keep the pixels with `(direction == k) & (mag >= front) & (mag >= back)`; copy their magnitude into a zero array `result`.
5. Return `result`.

#### `edges.hysteresis_threshold(image, low, high)` (used internally)

**What it is.** Two-threshold selection of edge pixels. Pixels above `high` are certain edges; pixels above `low` are accepted only if they are connected (through 8-neighbours above `low`) to a certain edge. This keeps long edges continuous while rejecting isolated weak responses (noise).

**Input.**
- `image` (array-like): thinned gradient magnitude.
- `low`, `high` (`float`): thresholds with `low <= high`.

**Output.** Boolean array (edge map). Raises `ValueError` if `low > high`.

**How to write it yourself.**
1. Check `low <= high`.
2. `weak = values > low` marks candidates.
3. Label the connected components of `weak` using 8-connectivity: `labels, count = ndimage.label(weak, structure=np.ones((3, 3)))`; every component gets a number 1, 2, …, `count`.
4. If `count == 0`, return an all-`False` array.
5. Find the labels of components containing a strong pixel: `np.unique(labels[values > high])`, without the background label 0.
6. Return `np.isin(labels, strong_labels)`: whole components that touch a strong pixel.

(The lecture notes grow edges iteratively along the edge direction; labelling connected components gives the same idea in a few vectorized lines.)

### Module `wavelets_lab.images` (file `src/wavelets_lab/images.py`)

Images as NumPy arrays and simple pixel operations. **Convention used everywhere in the course:** a greyscale image is a 2D array of type `float64` with values on the scale `[0, 255]` (0 = black, 255 = white); a colour image is a 3D array of shape `(rows, cols, 3)` on the same scale. Working in `float64` (instead of the `uint8` stored in image files) avoids overflow and rounding during computations.

The module starts with the imports
```python
from pathlib import Path
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
```
and the constant `MAX_INTENSITY = 255.0`.

#### `images.load_test_image(name="camera")`

**What it is.** Loads one of the licence-free test images shipped with scikit-image (`skimage.data`), so that no image file is needed.

**Input.**
- `name` (`str`, default `"camera"`): name of a function in `skimage.data`, e.g. `"camera"` (512×512 greyscale), `"astronaut"` (512×512 RGB), `"coins"`, `"checkerboard"`.

**Output.** `numpy.ndarray` of type `float64` on the scale `[0, 255]` (2D for greyscale, `(rows, cols, 3)` for colour). Raises `ValueError` if the name does not exist.

**How to write it yourself.**
1. Import the module of test images: `from skimage import data`.
2. Look the loader up by name without failing: `loader = getattr(data, name, None)`.
3. If `loader` is `None` or not callable (`callable(loader)` is `False`), raise `ValueError(f"skimage.data has no test image called {name!r}")`.
4. Call it: `raw = loader()` returns a `uint8` array.
5. Return `to_float(raw)` (see `images.to_float`), which converts to `float64` on `[0, 255]`.

#### `images.to_float(image)` (used internally)

**What it is.** Converts any image array to the course convention: `float64` on `[0, 255]`.

**Input.**
- `image` (array-like): `uint8`, `bool`, integer or floating-point image.

**Output.** `numpy.ndarray` of type `float64`.

**How to write it yourself.**
1. `array = np.asarray(image)`.
2. If `array.dtype == np.bool_`: return `array.astype(np.float64) * 255.0` (True becomes white).
3. If the dtype is floating (`np.issubdtype(array.dtype, np.floating)`), the array is not empty and all values lie in `[0, 1]` (`array.min() >= 0` and `array.max() <= 1`): the image uses the unit scale, so return `array.astype(np.float64) * 255.0`.
4. Otherwise (integers, or floats already on a large scale) return `array.astype(np.float64)`.

#### `images.show_image(image, title=None, ax=None, scaled=False, xlabel="column [pixel]", ylabel="row [pixel]")`

**What it is.** Displays a greyscale or colour image with Matplotlib, using a 256-level grey colour map and **labelled axes** (by default pixel coordinates: column on the horizontal axis, row on the vertical axis, with row 0 at the top).

**Input.**
- `image` (array-like): 2D greyscale or `(rows, cols, 3)` RGB image on `[0, 255]`.
- `title` (`str` or `None`): title of the plot.
- `ax` (`matplotlib.axes.Axes` or `None`): where to draw; when `None` a new figure is created. Passing `ax` lets you place several images in one figure (`fig, axes = plt.subplots(1, 3)`).
- `scaled` (`bool`, default `False`): `False` maps 0 to black and 255 to white (true intensities); `True` stretches the smallest value of the data to black and the largest to white, like Matlab's `imagesc`. Use `True` for arrays that are not intensities (spectra, gradients, phases).
- `xlabel`, `ylabel` (`str`): axis labels; change them when the axes mean something else (e.g. angle and offset for a sinogram).

**Output.** The `Axes` object that contains the image (useful for further changes such as `ax.set_xlim`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. If `ax is None`: `_, ax = plt.subplots()`.
3. If `array.ndim == 3` (colour): `ax.imshow(to_uint8(array))`; Matplotlib then uses the RGB values directly.
4. Otherwise (greyscale): choose limits `vmin, vmax = (None, None)` if `scaled` else `(0.0, 255.0)` and call `ax.imshow(array, cmap=plt.get_cmap("gray", 256), vmin=vmin, vmax=vmax)`. With `vmin=vmax=None` Matplotlib uses the data minimum and maximum.
5. If a title is given: `ax.set_title(title)`.
6. Label both axes: `ax.set_xlabel(xlabel)` and `ax.set_ylabel(ylabel)`.
7. Return `ax`.

#### `images.to_uint8(image)` (used internally)

**What it is.** The inverse direction: prepares an image for saving or for `imshow` of colour images, which expects bytes.

**Input.**
- `image` (array-like): image on the scale `[0, 255]`, possibly with values outside it.

**Output.** `numpy.ndarray` of type `uint8`.

**How to write it yourself.**
1. Convert to floats: `np.asarray(image, dtype=np.float64)`.
2. Round to the nearest integer with `np.rint`.
3. Clip to the valid range with `np.clip(..., 0, 255)`; without this, e.g. 260 would wrap around to 4.
4. Cast with `.astype(np.uint8)` and return.

### Module `wavelets_lab.timeseries` (file `src/wavelets_lab/timeseries.py`)

Multiscale analysis of financial time series (chapter 7 of the lecture notes): a series is split into additive components living on different time scales (years, months, days for daily data), and two series are compared scale by scale. Imports:
```python
import numpy as np
import pywt
```

#### `timeseries.synthetic_prices(n=2048, rng=None)`

**What it is.** A synthetic daily price series to experiment with before using real data: the logarithm of the price is `log 100 + 0.3 sin(2π d / 500) + 0.05 sin(2π d / 30) + random walk`, i.e. a slow cycle (about two years of trading days), a monthly cycle and day-to-day noise.

**Input.** `n` (`int`): number of days; `rng` (generator, seed or `None`).

**Output.** `float64` array of positive prices starting near 100.

**How to write it yourself.**
1. `generator = np.random.default_rng(rng)`, `days = np.arange(n)`.
2. Random walk: `np.cumsum(0.01 * generator.standard_normal(n))` (daily log-returns with standard deviation 1%).
3. `log_price = np.log(100) + 0.3 * np.sin(2 * np.pi * days / 500) + 0.05 * np.sin(2 * np.pi * days / 30) + random_walk`.
4. Return `np.exp(log_price)`.

#### `timeseries.multiscale_components(x, wavelet="db4", level=6)`

**What it is.** Splits a series into `level + 1` components, each reconstructed from **one** band of the DWT: the smooth trend (from the approximation) and one detail component per level. Because the DWT is linear, the components add up exactly to the series. Detail level `j` describes fluctuations on time scales of about `2^j` samples.

**Input.** `x` (array-like): series with length divisible by `2**level`; `wavelet` (`str`); `level` (`int`).

**Output.** List of `level + 1` arrays of the length of `x`: `[smooth, detail_level, …, detail_1]` (from coarse to fine).

**How to write it yourself.**
1. `coefficients = pywt.wavedec(x, wavelet, mode="periodization", level=level)`.
2. For every band index `i`: make a copy of the list in which all bands except band `i` are replaced by `np.zeros_like(band)`, and reconstruct it with `pywt.waverec(selected, wavelet, mode="periodization")`.
3. Return the list of the `level + 1` reconstructions.

#### `timeseries.group_scales(components, groups)`

**What it is.** Adds components into named groups of scales, e.g. "large" (trend and slowest details), "medium" and "small" scale, to get a few interpretable curves.

**Input.** `components` (list of arrays, from `multiscale_components`); `groups` (`dict[str, list[int]]`): name → indices of components, e.g. `{"large": [0, 1], "medium": [2, 3], "small": [4, 5, 6]}`.

**Output.** `dict[str, numpy.ndarray]`: name → sum of the listed components.

**How to write it yourself.** Return `{name: np.sum([components[i] for i in indices], axis=0) for name, indices in groups.items()}`.

#### `timeseries.scale_correlations(x, y, wavelet="db4", level=6)`

**What it is.** Correlates two series separately on every scale: both are split with `multiscale_components` and the Pearson correlation coefficient of each pair of corresponding components is computed. Two assets can be strongly related on long scales and unrelated day to day, or the other way round.

**Input.** `x`, `y` (array-like, same length, e.g. daily log-returns of two assets); `wavelet` (`str`); `level` (`int`).

**Output.** `float64` array of `level + 1` correlation coefficients in `[-1, 1]`, in the order of `multiscale_components`.

**How to write it yourself.**
1. `first = multiscale_components(x, wavelet, level)` and `second = multiscale_components(y, wavelet, level)`.
2. For each pair `(a, b)`: `np.corrcoef(a, b)[0, 1]`.
3. Return the values as an array.